## Cell 1 — Install Dependencies

In [1]:
import subprocess, sys

print("Installing dependencies...")
subprocess.run(["apt-get", "install", "-y", "curl"], check=False, capture_output=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "fastapi", "uvicorn", "pyngrok", "nest-asyncio", "httpx", "requests", "pydantic"],
    check=True,
)
print("Done.")


Installing dependencies...
Done.


## Cell 2 — Install & Start Ollama

In [2]:
import os, time, requests, subprocess

# Step 1 — install zstd (required by Ollama installer)
print("Installing zstd...")
subprocess.run(["apt-get", "install", "-y", "zstd"], check=True, capture_output=True)
print("zstd installed.")

# Step 2 — install Ollama
print("Installing Ollama...")
install = subprocess.run(
    "curl -fsSL https://ollama.com/install.sh | OLLAMA_SKIP_SYSTEMD=1 sh",
    shell=True, capture_output=True, text=True,
)
if install.returncode != 0:
    raise RuntimeError(
        f"Ollama install failed.\nSTDOUT:\n{install.stdout[-2000:]}\nSTDERR:\n{install.stderr[-2000:]}"
    )
print("Ollama installed.")

OLLAMA_BINARY = "/usr/local/bin/ollama"
if not os.path.exists(OLLAMA_BINARY):
    raise SystemExit("Ollama binary not found after install.")

# Step 3 — start server
print("Starting Ollama server...")
ollama_proc = subprocess.Popen(
    [OLLAMA_BINARY, "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

print("Waiting for Ollama...", end="", flush=True)
for attempt in range(60):
    try:
        if requests.get("http://localhost:11434/", timeout=2).status_code == 200:
            print(f" ready ({attempt + 1}s)")
            break
    except Exception:
        print(".", end="", flush=True)
        time.sleep(1)
else:
    raise SystemExit("Ollama did not start within 60 seconds.")

Installing zstd...
zstd installed.
Installing Ollama...
Ollama installed.
Starting Ollama server...
Waiting for Ollama..... ready (3s)


## Cell 3 — Pull Qwen3:8b Model

In [3]:
OLLAMA_MODEL = "qwen3:8b"

print(f"Pulling {OLLAMA_MODEL} (may take a few minutes on first run)...")
pull = subprocess.run(["ollama", "pull", OLLAMA_MODEL])
if pull.returncode != 0:
    raise RuntimeError(f"Failed to pull {OLLAMA_MODEL}")

model_names = [m["name"] for m in requests.get("http://localhost:11434/api/tags").json().get("models", [])]
if not any(OLLAMA_MODEL.split(":")[0] in n for n in model_names):
    raise RuntimeError(f"{OLLAMA_MODEL} not found after pull. Available: {model_names}")

print(f"{OLLAMA_MODEL} is ready. Available models: {model_names}")


Pulling qwen3:8b (may take a few minutes on first run)...
qwen3:8b is ready. Available models: ['qwen3:8b']


## Cell 4 — FastAPI App + /extract Endpoint

Receives a plain-text transcript from the local FastAPI service, runs it through
Qwen3:8b, and returns a structured `MedicalExtraction` JSON object.

In [4]:
import asyncio, json, re, time
import requests

import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException, Body
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field

nest_asyncio.apply()

OLLAMA_MODEL = "qwen3:8b"

app = FastAPI(title="MedAssist Ollama Extraction Service", version="1.0.0")


# ── Pydantic schemas ──────────────────────────────────────────────────────

class MedicineItem(BaseModel):
    name:         str = ""
    dose:         str = ""
    frequency:    str = ""
    duration:     str = ""
    instructions: str = ""


class MedicalExtraction(BaseModel):
    chief_complaints:           list[str]          = Field(default_factory=list)
    patient_reported_symptoms:  list[str]          = Field(default_factory=list)
    symptoms:                   list[str]          = Field(default_factory=list)
    past_conditions_mentioned:  list[str]          = Field(default_factory=list)
    conditions_mentioned:       list[str]          = Field(default_factory=list)
    medications_mentioned:      list[str]          = Field(default_factory=list)
    prescribed_medications:     list[MedicineItem] = Field(default_factory=list)
    body_parts_mentioned:       list[str]          = Field(default_factory=list)
    duration:                   list[str]          = Field(default_factory=list)
    severity:                   list[str]          = Field(default_factory=list)
    doctor_observations:        list[str]          = Field(default_factory=list)
    doctor_confirmed_diagnosis: list[str]          = Field(default_factory=list)
    advice:                     list[str]          = Field(default_factory=list)
    recommended_tests:          list[str]          = Field(default_factory=list)
    follow_up:                  list[str]          = Field(default_factory=list)
    risk_flags:                 list[str]          = Field(default_factory=list)
    uncertain_items:            list[str]          = Field(default_factory=list)


# ── Prompt ────────────────────────────────────────────────────────────────
# /nothink goes in the SYSTEM field — NOT in the user prompt body.
# Embedding it in the prompt text causes Ollama to 500 on some versions.

_SYSTEM = (
    "/nothink\n"
    "You are a clinical NLP assistant. Extract structured medical information "
    "from doctor-patient consultation transcripts. The transcript may be in "
    "Malayalam, English, or a mix of both.\n"
    "Return ONLY a valid JSON object — no preamble, no explanation, "
    "no markdown fences, no <think> blocks."
)

_SCHEMA = """
{
  "chief_complaints":           [],
  "patient_reported_symptoms":  [],
  "symptoms":                   [],
  "past_conditions_mentioned":  [],
  "conditions_mentioned":       [],
  "medications_mentioned":      [],
  "prescribed_medications": [
    {
      "name": "",
      "dose": "",
      "frequency": "",
      "duration": "",
      "instructions": ""
    }
  ],
  "body_parts_mentioned":        [],
  "duration":                    [],
  "severity":                    [],
  "doctor_observations":         [],
  "doctor_confirmed_diagnosis":  [],
  "advice":                      [],
  "recommended_tests":           [],
  "follow_up":                   [],
  "risk_flags":                  [],
  "uncertain_items":             []
}"""

_USER_TEMPLATE = (
    "Extract medical information from this transcript and return ONLY the JSON "
    "object matching this exact schema:\n{schema}\n\nTRANSCRIPT:\n{transcript}"
)


# ── JSON extraction helper ────────────────────────────────────────────────

def _extract_json_object(text: str) -> str:
    """
    Robustly extract the first complete top-level {...} JSON object.

    Handles: clean JSON, markdown fences, leading/trailing <think> blocks,
    trailing prose after the closing brace.
    """
    # 1. Strip <think>...</think> blocks anywhere in the output
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

    # 2. Strip markdown fences on first/last line
    lines = text.splitlines()
    if lines and lines[0].startswith("```"):
        lines = lines[1:]
    if lines and lines[-1].strip() == "```":
        lines = lines[:-1]
    text = "\n".join(lines).strip()

    # 3. Walk chars to extract outermost {...} — ignores trailing prose
    depth, start = 0, None
    for i, ch in enumerate(text):
        if ch == "{":
            if depth == 0:
                start = i
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0 and start is not None:
                return text[start : i + 1]

    return text   # fallback — json.loads will raise a clear error


# ── Ollama call ───────────────────────────────────────────────────────────

def _call_ollama_raw(transcript: str) -> str:
    """Return the raw string from Ollama with no parsing. Used by /debug-raw."""
    payload = {
        "model":  OLLAMA_MODEL,
        "system": _SYSTEM,
        "prompt": _USER_TEMPLATE.format(
                      schema=_SCHEMA,
                      transcript=transcript.strip()),
        "stream": False,
        "options": {"temperature": 0, "num_predict": 2048},
    }
    resp = requests.post(
        "http://localhost:11434/api/generate",
        json=payload,
        timeout=120,
    )
    # Log Ollama-level errors with full body for easy diagnosis
    if resp.status_code != 200:
        print(f"[Ollama ERROR {resp.status_code}] {resp.text[:600]}")
    resp.raise_for_status()
    return resp.json().get("response", "")


def _call_ollama(transcript: str) -> dict:
    """Synchronous Ollama call — runs in a thread via asyncio.to_thread."""
    raw = _call_ollama_raw(transcript)
    print(f"[Ollama raw first 300] {repr(raw[:300])}")
    clean = _extract_json_object(raw.strip())
    print(f"[Ollama clean first 200] {repr(clean[:200])}")
    return json.loads(clean)


# ── Health check ──────────────────────────────────────────────────────────

@app.get("/")
def health():
    return {"status": "ok", "model": OLLAMA_MODEL, "endpoint": "/extract"}


# ── POST /debug-raw ───────────────────────────────────────────────────────

@app.post("/debug-raw")
async def debug_raw(
    transcript: str = Body(..., embed=True,
                           description="Returns raw Ollama output before parsing"),
):
    """
    POST { "transcript": "..." }  →  raw Ollama string + parse diagnostics.
    Use this to diagnose 500 errors on /extract.
    """
    if not transcript.strip():
        raise HTTPException(status_code=422, detail="transcript must not be empty")
    try:
        raw = await asyncio.to_thread(_call_ollama_raw, transcript)
    except Exception as exc:
        raise HTTPException(status_code=500, detail=f"Ollama call failed: {exc}")

    clean = _extract_json_object(raw.strip())
    parse_ok, parse_error = True, None
    try:
        json.loads(clean)
    except json.JSONDecodeError as exc:
        parse_ok, parse_error = False, str(exc)

    return JSONResponse({
        "raw_response": raw,
        "cleaned":      clean,
        "parse_ok":     parse_ok,
        "parse_error":  parse_error,
    })


# ── POST /extract ─────────────────────────────────────────────────────────

@app.post("/extract")
async def extract(
    transcript: str = Body(..., embed=True,
                           description="Transcript text (Malayalam / English / mixed)"),
):
    """
    POST  { "transcript": "..." }
    Returns structured MedicalExtraction JSON extracted by Qwen3:8b.
    """
    if not transcript.strip():
        raise HTTPException(status_code=422, detail="transcript must not be empty")

    start = time.perf_counter()

    try:
        data = await asyncio.to_thread(_call_ollama, transcript)
    except json.JSONDecodeError as exc:
        print(f"[JSONDecodeError] {exc}")
        raise HTTPException(status_code=500, detail=f"Ollama returned invalid JSON: {exc}")
    except Exception as exc:
        print(f"[ExtractError] {type(exc).__name__}: {exc}")
        raise HTTPException(status_code=500, detail=f"Extraction error: {exc}")

    try:
        extraction = MedicalExtraction(**data)
    except Exception as exc:
        print(f"[ValidationError] {exc}")
        raise HTTPException(status_code=500, detail=f"Schema validation failed: {exc}")

    elapsed = round(time.perf_counter() - start, 2)
    print(f"[/extract] OK {elapsed}s | complaints={extraction.chief_complaints[:2]}")

    return JSONResponse({
        "success":             True,
        "processing_time_sec": elapsed,
        "extraction":          extraction.model_dump(),
    })


print("FastAPI app defined.")
print("  GET  /             — health check")
print("  POST /extract      — transcript → MedicalExtraction")
print("  POST /debug-raw    — transcript → raw Ollama output (diagnosis)")


FastAPI app defined.
  GET  /         — health check
  POST /extract  — transcript → MedicalExtraction


## Cell 5 — Start uvicorn + ngrok

Paste your ngrok auth token into `NGROK_TOKEN` before running.  
Get one free at https://dashboard.ngrok.com/get-started/your-authtoken

In [5]:
import threading
from pyngrok import ngrok

# ⚠️  Paste your token here. Do NOT commit or share it.
NGROK_TOKEN = ""
if not NGROK_TOKEN:
    raise ValueError("Paste your ngrok token into NGROK_TOKEN before running this cell.")

ngrok.set_auth_token(NGROK_TOKEN)

def _run():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

threading.Thread(target=_run, daemon=True).start()
time.sleep(2)

tunnel = ngrok.connect(8000)
public_url = tunnel.public_url  # FIX: extract the string URL from the NgrokTunnel object

print("=" * 60)
print("MEDASSIST OLLAMA EXTRACTION SERVICE IS LIVE")
print("=" * 60)
print(f"Base URL  : {public_url}")
print(f"Health    : {public_url}/")
print(f"Extract   : {public_url}/extract")
print(f"Debug     : {public_url}/debug-raw")
print("=" * 60)
print(f'\nSet this in your local config.py / .env:')
print(f'COLAB_BASE_URL = "{public_url}"')
print("=" * 60)


MEDASSIST OLLAMA EXTRACTION SERVICE IS LIVE
Base URL  : NgrokTunnel: "https://repair-dolly-dollop.ngrok-free.dev" -> "http://localhost:8000"
Health    : NgrokTunnel: "https://repair-dolly-dollop.ngrok-free.dev" -> "http://localhost:8000"/
Extract   : NgrokTunnel: "https://repair-dolly-dollop.ngrok-free.dev" -> "http://localhost:8000"/extract

Set this in your local config.py / .env:
COLAB_BASE_URL = "NgrokTunnel: "https://repair-dolly-dollop.ngrok-free.dev" -> "http://localhost:8000""


## Cell 6 — Keep-Alive Loop

Prevents the Colab session from timing out. **Stop this cell when you are done.**

In [ ]:
import datetime

print("Keep-alive started. Stop this cell when finished.")
counter = 0
while True:
    counter += 1
    print(f"[{datetime.datetime.now().strftime('%H:%M:%S')}] alive — {counter * 5} min elapsed")
    time.sleep(300)


Keep-alive started. Stop this cell when finished.
[10:11:02] alive — 5 min elapsed
[10:16:02] alive — 10 min elapsed
